# Notebook 05 — Time Series Forecasting
**Contributor:** Dingaan Mahlatse Machethe (SKYLearn-Innovation head — MSc Data Science, UEL)  
**Tier:** MASTERY  

We forecast future matric pass rates using **Holt's Exponential Smoothing (ETS)** with an
additive trend. For short annual series (10 points) ETS is more stable than ARIMA.
We produce point forecasts with 95% confidence intervals and rank subjects by projected risk.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

from preprocess import load_clean
from forecast import forecast_series, forecast_all_subjects

plt.style.use('seaborn-v0_8-whitegrid')
df = load_clean()
print('Dataset:', df.shape)

## 1. Forecast a single series — National Mathematics

In [ ]:
res = forecast_series(df, subject='Mathematics', horizon=3)
hist, fc = res['history'], res['forecast']

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(hist['year'], hist['pass_rate'], 'o-', color='#1F4E79', linewidth=2.5, label='History')
ax.plot(fc['year'], fc['forecast'], 's--', color='#E74C3C', linewidth=2.5, label='Forecast')
ax.fill_between(fc['year'], fc['lower'], fc['upper'], color='#E74C3C', alpha=0.15, label='95% CI')
ax.axhline(60, color='gray', linestyle=':', label='60% benchmark')
ax.set_title(f"{res['label']} — 3-Year Forecast ({fc['method'].iloc[0]})", fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Pass Rate (%)'); ax.legend()
plt.tight_layout(); plt.show()
fc

## 2. Compare forecasts for all STEM subjects (national)

In [ ]:
stem_subjects = df[df['subject_type'] == 'STEM']['subject'].unique()
fig, ax = plt.subplots(figsize=(11, 6))
for subj in stem_subjects:
    r = forecast_series(df, subject=subj, horizon=3)
    full_years = list(r['history']['year']) + list(r['forecast']['year'])
    full_vals = list(r['history']['pass_rate']) + list(r['forecast']['forecast'])
    ax.plot(full_years, full_vals, 'o-', linewidth=1.8, label=subj)
ax.axvline(df['year'].max(), color='gray', linestyle=':', alpha=0.6)
ax.text(df['year'].max()+0.1, 45, 'forecast →', color='gray')
ax.set_title('STEM Subjects — History + 3-Year Forecast', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Pass Rate (%)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Early-warning ranking — which subjects need intervention?

In [ ]:
ranked = forecast_all_subjects(df, horizon=3)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E74C3C' if v < 60 else '#117A65' for v in ranked['forecast_pass_rate']]
ax.barh(ranked['subject'], ranked['forecast_pass_rate'], color=colors)
ax.axvline(60, color='gray', linestyle='--')
ax.set_title(f"Projected {ranked['forecast_year'].iloc[0]} Pass Rate by Subject", fontweight='bold')
ax.set_xlabel('Forecast Pass Rate (%)')
plt.tight_layout(); plt.show()
ranked

## Key Findings
1. **Mathematics** is forecast to remain the most at-risk subject, projected below the 60% benchmark
2. Confidence intervals widen with horizon — uncertainty compounds, as expected
3. The forecasting engine powers the live dashboard's 🔮 Forecast page
4. **Action:** subjects forecast below 60% should receive targeted SKYLearn-Innovation tutoring